# Explicación de la tabla y sus campos.

La tabla muestra un préstamo por registro. No presenta una clave primaria para distinguir cada préstamo, por ello he establecido como clave el conjunto de campos prbarc, prprsu, prfpre y prlebi. Que a continuación detallo.

Campos presentes:

prbarc: Corresponde al código numérico único que distingue al ejemplar/material prestado. No nulo.

prprsu: Código alfabético de la biblioteca pública donde se ha ejecutado el préstamo. Cada biblioteca tiene su propio código. Si código nulo no se puede identificar la biblioteca. Hay 4 valores nulos en el campo. Se eliminan al ser este campo parte de la clave.

prcolp: Código alfabético que identifica el tipo de lector que ha realizado el préstamo. Categorización amplia, engloba tipos como lector, biblioteca, usuario-ebook...

prcocp: Código alfabético del tipo de ejemplar prestado. Se filtrará la tabla por los registros dentro de las siguientes categorías: Prestables Libros - código prl

prlebi: Código alfabético que identifica la red de bibliotecas donde el usuario realizó el carnet. Puede ser BPM (biblioteca pública de Madrid) u otros.

prlesu: Código alfabético que identifica la biblioteca pública donde el usuario realizó el carnet. Mismos códigos que prprsu.

pradul: Código binario que señala si el carnet es de tipo adulto o no, siendo la alternativa carnet infantil.

prfpre: Fecha en la que se realizó el préstamo. No hay nulos.

prcocs: Código alfabético del tipo de soporte del ejemplar prestado. La tabla se lee filtrando por aquellos registros que sean libros (lib), no hay más tipos dentro del objeto de análisis.

tititu: Título del ejemplar prestado, cubre hasta 60 caracteres del título original.

tiauto: Autor del ejemplar prestado.

tiisxn: Código ISBN. Puede haber más de un ejemplar con el mismo ISBN.

prcoc1: Código alfabético que señala público destinatario objetivo del ejemplar (ej. adulto, juvenil...)

Serán objeto del análisis los préstamos realizados en una biblioteca real (se eliminan préstamos ejecutados en códigos de biblioteca que no pertenecen a un centro bibliotecario abierto al público general), a los carnés de tipo individual (prcolp) y materiales libros (prcocs = lib y prcocp = prl).


# Acciones iniciales

In [ ]:
from google.colab import drive # montar drive
import shutil # copiar docs. drive en disco local
from pathlib import Path # Gestionar rutas archivos
import json # Para crear y tratar archivo json
import pandas as pd
import re # módulo de expresiones regulares
import numpy as np
import gc
from IPython.display import display

pd.options.mode.copy_on_write = True
pd.set_option("mode.string_storage", "pyarrow")
pd.set_option("io.parquet.engine", "pyarrow")

In [ ]:
# Montar drive:
drive.mount('/content/drive')

## Constantes

In [ ]:
# CONSTANTES TRATAMIENTO ARCHIVOS

# 1. Rutas originales
pf = Path('drive')
local = Path('/content')

# 2. Rutas sobre las que trabajar:
# 2.1 Rutas que apuntan hacia Drive
DRIVE_ORIGEN = pf / 'bronze'
DRIVE_DESTINO = pf / 'silver'

# 2.2 Rutas que apuntan hacia disco local:
ORIGEN_LOCAL = local / 'datos_bronze'
DESTINO_LOCAL = local / 'resultados_silver'
ORIGEN_LOCAL.mkdir(parents=True, exist_ok=True)
DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)


# 3. Limpieza y transformaciones DF
PATRONES_NULOS = ("", "-", ".", "?", "n/a", "na", "n.a", "n.a.",
                  "null", "none", "nan","sd", "s/d", "sin datos",
                  "tbd", "x", "xx", "xxx")

## Copia archivos drive en local

In [ ]:
# Copiar todas las subcarpetas de origen drive
#pudiendo excluir algunas.

EXCLUIR = ["PRESTAMOS_HISTORICOS"]
assert DRIVE_ORIGEN.exists(), f"No existe {DRIVE_ORIGEN}"

disponibles = sorted(p.name for p in DRIVE_ORIGEN.iterdir() if p.is_dir())
seleccion = [s for s in disponibles if s not in EXCLUIR]

faltan = [s for s in seleccion if s not in disponibles]
assert not faltan, f"No existen en DRIVE_ORIGEN: {faltan}. Disponibles: {disponibles}"

for sub in seleccion:
    shutil.copytree(DRIVE_ORIGEN / sub, ORIGEN_LOCAL / sub, dirs_exist_ok=True)

DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)

print(f"Subcarpetas en Drive: {disponibles}")
print(f"Copiadas: {seleccion}\n")

for sub in seleccion:
    csvs = sorted(p.name for p in (ORIGEN_LOCAL / sub).rglob("*.csv"))
    print(f"{sub} ({len(csvs)} csv): {csvs}")

In [ ]:
# Importación archivo JSON desde Drive:

jsons = list(DRIVE_ORIGEN.glob('*.json'))
assert jsons, f'No hay ningún .json en {DRIVE_ORIGEN}'
for f in jsons:
    shutil.copy2(f, ORIGEN_LOCAL / f.name)
print(f"JSON copiados: {[f.name for f in jsons]}")


# Lectura archivo JSON para emplearlo en lectura CSV:

with open(ORIGEN_LOCAL / 'parametros_csv_brutos.json', encoding='utf-8') as f:
    params_json = json.load(f)

# Lectura CSV - PRESTAMOS

In [ ]:
df_prestamos, resumen = lectura_prestamos(params_json, 'PRESTAMOS',
                                          lista, ORIGEN_LOCAL)

CSV de préstamos leídos: 101
DF final: 18730236 filas y 13 columnas
Filas descartadas por el filtro: 4334932



,antes,despues,descartadas,lineas_reparadas,lineas_desc_objetivo,lineas_desc_otras,filas_cortas,isbn_cientifico,isbn_raros,cambios
2018_02_febrero,294378,189275,105103,0,0,0,0,<NA>,<NA>,
2018_03_marzo,315284,205888,109396,0,0,0,0,<NA>,<NA>,
2018_04_abril,318494,215189,103305,0,0,0,0,<NA>,<NA>,
2018_05_mayo,280361,189036,91325,0,0,0,0,<NA>,<NA>,
2018_06_junio,268819,184080,84739,0,0,0,0,<NA>,<NA>,
...,...,...,...,...,...,...,...,...,...,...
2026_05_mayo,247163,219057,28106,0,0,0,0,0,96,"quotechar '→"""
2026_06_junio,240811,214797,26014,0,0,0,0,0,133,"quotechar '→"""
2026_07_julio,244881,220085,24796,0,0,0,0,0,131,
2026_08_agosto,214343,191192,23151,0,0,0,0,0,149,



Archivos con líneas reparadas o descartadas: ['2021_08_agosto', '2024_11_noviembre', '2024_12_diciembre', '2025_01_enero', '2025_02_febrero', '2025_03_marzo', '2025_04_abril', '2025_07_julio', '2025_09_septiembre', '2025_10_octubre', '2025_11_noviembre', '2026_02_febrero', '2026_03_marzo', '2026_04_abril']


In [ ]:
#Quitar comillas títulos:
df_prestamos, filas_comillas = quitar_comillas(df_prestamos)
print(f"Filas con comillas eliminadas: {filas_comillas}")

# Corregir doble codificación
df_prestamos, filas_mojibake = corregir_mojibake(df_prestamos)
# Comprobación corrección
for col in COLUMNAS_TEXTO:
    quedan = df_prestamos[col].str.contains("[ÃÂ]", regex=True).fillna(False)
    print(col, int(quedan.sum()))
    display(df_prestamos.loc[quedan, col].drop_duplicates().head(10))


Filas con comillas eliminadas: {'tititu': 42384, 'tiauto': 1139}
tititu 107


,tititu
57719,La felicidad paradÂojica : ensayo sobre la soc...
95816,Fumata blanca : la elecciÂon de Benedicto XVI ...
132686,"Âromm. 1, Destino nómada"
284022,AntropologÂia del cerebro : la conciencia y lo...
287683,Biblia y helenismo : el pensamiento griego y l...
635921,"Âromm. 2, Corazón de estepa"
911731,"Los perros de Dios : ColÂon, la InquisiciÂon y..."
1815636,L'Art au Moyen Âge
3072525,La ville au Moyen-Âge
3721903,Petit Âne veut être un loup


tiauto 1237


,tiauto
95816,"GalÂan, Lola"
222402,"Masot, NÂuria (1949-)"
254142,"Madeira, Ângela"
438561,SjÂon (1962-)
485127,"GarcÂia Madruga, Juan Antonio"
546436,"GarcÂia-RomÂeu, Carmen"
1378184,"Marsak, Samuil Âkovlevi (1887-1964)"
1737705,"Ruiz Domenec, JosÂe Enrique"
8427805,"DÃ¥snes, Nora (1995-)"
8446312,"Tusquets, Ã’scar (1941-)"


# Limpieza CSV - PRESTAMOS

Antes de realizar la limpieza general, se pasan a tipo category, las columnas vistas en las tablas soporte de préstamos, con tal de reducir tamaño de las mismas y reducir carga de memoria.

In [ ]:
#Poner tipo category a campos:
campos_category = ['prcocp', 'prprsu', 'prcolp', 'prlebi', 'prlesu',
                   'pradul', 'prcocs', 'prcoc1']

for col in campos_category:
    df_prestamos[col] = df_prestamos[col].astype('category')
    gc.collect()

In [ ]:
df_prestamos, df_resumen_nulos = transformaciones_comunes(df_prestamos)
gc.collect()


--------------- DF: PRESTAMOS HISTORICOS ---------------

1. Información incial:

1.1 Muestra:



,prbarc,prprsu,prcolp,prcocp,prlebi,prlesu,pradul,prfpre,prcocs,tititu,tiauto,tiisxn,prcoc1
609330,2378421,BP11,LEC,PRL,BPM,BP11,1,04/04/2018,LIB,Luces del norte. 1: basado en la obra de Phili...,"Melchior-Durand, Stéphane (1965-)",<NA>,<NA>



1.2 Información general:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18730236 entries, 0 to 18730235
Data columns (total 13 columns):
 #   Column  Dtype   
---  ------  -----   
 0   prbarc  string  
 1   prprsu  category
 2   prcolp  category
 3   prcocp  category
 4   prlebi  category
 5   prlesu  category
 6   pradul  category
 7   prfpre  string  
 8   prcocs  category
 9   tititu  string  
 10  tiauto  string  
 11  tiisxn  string  
 12  prcoc1  category
dtypes: category(8), string(5)
memory usage: 2.3 GB

2. Resultados post normalizaciones del DF:
2.1 Muestra:



,prbarc,prprsu,prcolp,prcocp,prlebi,prlesu,pradul,prfpre,prcocs,tititu,tiauto,tiisxn,prcoc1
13234388,101595262,bcdi5,lec,prl,bpm,bp13,1,13/08/2024,lib,carta al padre y otros escritos /,"kafka, franz (1883-1924)",84-206-3834-x,<NA>
10932909,7664103,bp21,lec,prl,bpm,bp06,1,18/09/2023,lib,el libro de los libros : historias sobre image...,"buchholz, quint",978-84-16830-15-2,<NA>



3. Primera inspección posibles nulos de la tabla:


Resumen posibles nulos encontrado:



,prprsu,prcolp,prlebi,prlesu,pradul,tititu,tiauto,tiisxn,prcoc1,total
'',4,132,132,144003,132,43,889410,166514,2948,1203318
'.',0,0,0,0,0,0,0,4,0,4
'x',0,0,0,0,0,23,0,0,0,23
'xxx',0,0,0,0,0,0,0,0,3,3
total,4,132,132,144003,132,66,889410,166518,2951,1203348


179

# Limpiezas particulares

In [ ]:
# Valores iniciales antes de las transformaciones:
nulos_e_i = df_prestamos.isna().sum().sum()
print(f"Las dimensiones inciales del df son de {df_prestamos.shape}."
f"\nLos posibles valores nulos totales encontrados son: {df_resumen_nulos.loc['total', 'total']}")
print(f"Valores nulos explicitos: {nulos_e_i}")
print(df_prestamos.dtypes)

Las dimensiones inciales del df son de (18730236, 13).
Los posibles valores nulos totales encontrados son: 1203348
Valores nulos explicitos: 18666939
prbarc    string[pyarrow]
prprsu           category
prcolp           category
prcocp           category
prlebi           category
prlesu           category
pradul           category
prfpre    string[pyarrow]
prcocs           category
tititu    string[pyarrow]
tiauto    string[pyarrow]
tiisxn    string[pyarrow]
prcoc1           category
dtype: object


Al inicio de la limpieza hay 1.203.348 posibles valores nulos, como se observa en la tabla resumen resultante de transformaciones comunes. Hay que extraer de ellos los registrados por un valor "xxx" en el campo prcoc1 ya que pertenecen a una categoría de público destinatario del libro. Quedando 1.203.345 posibles valores nulos. La gran mayoría corresponden a valores vacíos 1.203.318 cosa que no debería suceder, ya que los registros deberían presentar valor para los distintos campos, menos los registros anteriores a julio 2025 en el campo prcoc1 que no se registraban.

Se pasan a valor nulo explícito, excluyendo valor "xxx"

De las transformaciones resultantes a la hora de adaptar la lectura del formato de los distintos csv se han generado 18.666.939 valores nulos explícitos entre los campos tiisxn y prcoc1, ya que han sido los campos afectados por el doble encomillado y saltos de lineas.

## Conversión a nulo explícito

In [ ]:
total_vacios = df_resumen_nulos.loc["total", "total"] - 3
# Conversión a nulo explícito
print(f"Total valores vacios, en todos los campos, a convertir en nulo: "
f"{total_vacios}")
print(f"\nTotal valores nulos iniciales : {nulos_e_i}")

nulos_convertir = ["", ".", "x"]
campos_string = ['tititu', 'tiauto', 'tiisxn']
campos_category = ['prprsu', 'prcolp', 'prlebi', 'prlesu', 'pradul', 'prcoc1']

# campos formato string:
df_prestamos[campos_string] = df_prestamos[campos_string].replace(
    nulos_convertir, pd.NA)

nulos_string = df_prestamos.isna().sum().sum()
print(f"\nTotal valores nulos post conversion campos string: "
f"{nulos_string}")

# campos formato category:
for c in df_prestamos.select_dtypes("category").columns:
    presentes = [v for v in nulos_convertir if v in df_prestamos[c].cat.categories]
    df_prestamos[c] = df_prestamos[c].cat.remove_categories(presentes)

nulos_category = df_prestamos.isna().sum().sum()
print(f"\nTotal valores nulos post conversion campos category: "
f"{nulos_category}")

Total valores vacios, en todos los campos, a convertir en nulo: 1203345

Total valores nulos iniciales : 18666939

Total valores nulos post conversion campos string: 19722933

Total valores nulos post conversion campos category: 19870284


Total valores vacíos, en todos los campos, a convertir en nulo: 1203345

Total valores nulos iniciales : 18666939

Total valores nulos post conversión campos string: 19722933

Total valores nulos post conversión campos category: 19870284

## Pasar tipos de dato a formato adecuado

### Campo fecha - prfpre

En revisión inicial se ve que tiene formato d/m/año. En exploraciones posteriores se han observado diferentes tipos de formato. Se analizan los posibles formatos y se convierten según sea el caso del registro.

El campo viene en formato string, normalizado y sin espacios. No hay nulos en él.

Se crea nuevo campo de fecha para poder trabajar sobre él

In [ ]:
# Nuevo campo fecha bruta en df:
df_prestamos['fecha_bruta'] = df_prestamos['prfpre']

# Se trabaja sobre la serie generada:
s = df_prestamos["fecha_bruta"]

# Se busca patrones de tipo fecha
s.str.replace(r"\d", "9", regex=True).value_counts(dropna=False)

,count
fecha_bruta,
99/99/9999,15303127
99/99/99,1472757
9999-99-99 99:99:99,1337964
99/99/9999 99:99,590235
99/99/9999 9:99,26153


Se ve que hay 5 formatos diferentes entre solo fecha con año en 4 y 2 dígitos, fecha con años en 4 y 2 dígitos con hora y formato ISO.

Se establece el formato de referencia como solo fecha, entendiendo que los préstamos que se puedan realizar en un mismo día por las claves definidas -> código de barras (prbarc), mismo código de biblioteca por centro que realiza el préstamo (prprsu), mismo centro de emisión del carnet (prlesu) coinciden, código tipo lector (prcolp) y si el tipo de carnet es adulto o no (pradul), definen a un mismo usuario y si se muestran duplicados según ella es por invalidez de uno de los registros de préstamos o fallo de la realización del préstamo.

Se definen diferentes categorías según sea formato ISO, año 2 dígitos y año 4 dígitos, incluyéndose sean o no con hora.

Se filtran registros según condición y se convierte tipo según el formato filtrado.

In [ ]:
cats = {
    "iso":  s.str.fullmatch(r"\d{4}-\d{2}-\d{2}(\s+\d{1,2}:\d{2}(:\d{2})?)?", na=False),
    "dm4":  s.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}(\s+\d{1,2}:\d{2}(:\d{2})?)?", na=False),
    "dm2":  s.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{2}(\s+\d{1,2}:\d{2}(:\d{2})?)?", na=False),
}
cubierto = pd.concat(cats, axis=1).any(axis=1)
otros = ~cubierto & s.notna()

print({k: v.sum() for k, v in cats.items()}, "nulos:", s.isna().sum(), "otros:", otros.sum())
print(s[otros].value_counts().head(20))

{'iso': np.int64(1337964), 'dm4': np.int64(15919515), 'dm2': np.int64(1472757)} nulos: 0 otros: 0
Series([], Name: count, dtype: int64[pyarrow])


Se ven 1.337.964 para el formato ISO.

15.919.515 para el formato con año 4 dígitos.

1.472.757 para el formato con 2 dígitos.


In [ ]:
# Se quita la hora de los registros en los que esta presente:
s_fecha = s.str.replace(r"\s+\d{1,2}:\d{2}(:\d{2})?$", "", regex=True)

# Se pasa cada categoría de filtro a dtype Datetime según formato encontrado
f_iso = pd.to_datetime(s_fecha.where(cats["iso"]), format="%Y-%m-%d", errors="coerce")
f_dm4 = pd.to_datetime(s_fecha.where(cats["dm4"]), format="%d/%m/%Y", errors="coerce")
f_dm2 = pd.to_datetime(s_fecha.where(cats["dm2"]), format="%d/%m/%y", errors="coerce")

# Se asigna tipo de dato a la fecha original
df_prestamos["prfpre"] = f_iso.fillna(f_dm4).fillna(f_dm2)

# Se comprueba si nulos en la columna:
nulos_fecha = df_prestamos['prfpre'].isna().sum()
print(f"Han aparecido {nulos_fecha} valores nulos en el campo fecha prfpre")

Han aparecido 0 valores nulos en el campo fecha prfpre


In [ ]:
# Se miran si aparecen fechas imposibles

invalidas = cubierto & df_prestamos["prfpre"].isna()
print("Fechas imposibles:", invalidas.sum(), s[invalidas].unique()[:20])

# Se mira valores máximo y mínimos de fecha:
print(df_prestamos["prfpre"].min(), df_prestamos["prfpre"].max())

Fechas imposibles: 0 <ArrowStringArray>
[]
Length: 0, dtype: string
2018-02-01 00:00:00 2026-08-31 00:00:00


Fechas correctas, comprendidas dentro del periodo a analizar.

In [ ]:
# Se cancela campo fecha bruta:
del df_prestamos['fecha_bruta']

# Se cancelan variables generadas:

del s, cats, cubierto, otros, s_fecha, f_iso, f_dm2, f_dm4, invalidas

### Duplicados completos

In [ ]:
# Se mira cantidad de grupos de registros completamente duplicados:

duplis_completos = df_prestamos[df_prestamos.duplicated(keep=False)]

tam = duplis_completos.value_counts(dropna=False) # Duplicados teniendo en cuenta nulos
n_grupos = len(tam)

print(f"Numero de grupos registros duplicados: {len(tam)}\n"
f"\nGrupos de duplicados con más de dos registros {len(tam[tam > 2])}")

ej_eliminar = tam.sum() - len(tam)

print(f"\nSe eliminaran {ej_eliminar} registros."
f"Dejando {len(df_prestamos) - ej_eliminar} registros restantes")


Numero de grupos registros duplicados: 466852
Grupos de duplicados con más de dos registros 5989

Se eliminaran 475373 registros.Dejando 18254863 registros restantes


Número de grupos registros duplicados: 466852
Grupos de duplicados con más de dos registros 5989

Se eliminarán 475.373 registros. De los cuales 92.284 registros pertenecen a la duplicidad completa de abril'25, catalogada en fuente como mayo'25.

Dejando 18254863 registros restantes

In [ ]:
# Eliminación de uno de los registros:
print(f"filas iniciales:{len(df_prestamos)}")
df_prestamos = df_prestamos.drop_duplicates()
print(f"Num reg eliminados {ej_eliminar}")
ej_finales = len(df_prestamos)
print(f"Reg finales totales {ej_finales}")
del duplis_completos, tam, n_grupos
gc.collect()

filas iniciales:18730236
Num reg eliminados 475373
Reg finales totales 18254863


### Duplicados parciales

Primero se mira cuántos códigos de barras se ven repetidos un mismo día, con mismo código de biblioteca por centro que realiza el préstamo (prprsu), cuyo centro de emisión del carnet (prlesu) coinciden,  mismo código tipo lector (prcolp) y si el carnet es de tipo adulto o no (pradul)

En segunda fase, se mirará eliminando las claves prlesu, prcolp y pradul que son con las que se puede llegar a identificar muy parcialmente al usuario y ver así si hay duplicidades de ejemplar prestado por biblioteca y día.


La idea es ver si se ha podido realizar un préstamo a un tipo de carnet y que por algún motivo no haya podido ejecutarse y acabar viendo si se han realizado préstamos iguales (mismo código, día y centro)

In [ ]:
# Duplicados en clave:
clave  = ['prbarc','prprsu', 'prfpre','prlesu', 'prcolp', 'pradul' ]

duplis_clave = df_prestamos[df_prestamos.duplicated(subset=clave, keep=False)]
n_duplis_calve = len(duplis_clave)

if n_duplis_calve == 0:
  print("No hay registros duplicados según la clave")
else:
  tam2 = duplis_clave.value_counts(subset= clave,dropna=False)
  print(f"Se han encontrado {n_duplis_calve} registros que comparten"
        f" la clave - {clave}.\n"
        f"Repartidos en {len(tam2)} grupos"
        f"\nGrupos de duplicados con más de dos registros {len(tam2[tam2 > 2])}")
  pr_eliminar = tam2.sum() - len(tam2)
  print(f"\nSe eliminaran {pr_eliminar} registros."
  f"Dejando {len(df_prestamos) - pr_eliminar} registros restantes")

Se han encontrado 4271 registros que comparten la clave - ['prbarc', 'prprsu', 'prfpre', 'prlesu', 'prcolp', 'pradul'].
Repartidos en 2135 grupos
Grupos de duplicados con más de dos registros 1

Se eliminaran 2136 registros.Dejando 18252727 registros restantes


In [ ]:
duplis_clave.value_counts(dropna=False).head(6)

prbarc     prprsu  prcolp  prcocp  prlebi  prlesu  pradul  prfpre      prcocs  tititu                                        tiauto                              tiisxn                              prcoc1
100092089  bcdin   lec     prl     bpm     bp11    1       2025-04-08  lib     soledades                                     galerias                            otros poemas /                      NaN       1
                                                                               soledades ; galerias ; otros poemas /         machado, antonio (1875-1939)        978-84-376-0411-4                   NaN       1
100173361  bcdiu   lec     prl     bpm     bp16    1       2025-04-10  lib     el concierto de san ovidio                    la fundacion /                      buero vallejo, antonio (1916-2000)  NaN       1
                                                                               el concierto de san ovidio ; la fundacion /   buero vallejo, antonio (1916-2000)  84-239-1569-7                       NaN       1
100180789  bcdik   lec     prl     bpm     bp10    1       2025-04-07  lib     como me hice monja                            la prueba                           el llanto /                         NaN       1
                                                                               como me hice monja ; la prueba ; el llanto /  aira, cesar (1949-)                 84-397-0223-x                       NaN       1
Name: count, dtype: int64

Al observar duplicados por clave nos encontramos con 4.271 registros en grupos de dos (menos 1) que comparten clave. Sin embargo, se puede apreciar por la muestra extraída que estas duplicidades son, aparentemente, causadas por la decodificación de los archivos, ya que se observan valores erróneos por ISBN (tiisxn) mal codificado (con texto perteneciente al título) en uno de los registros y la versión correcta en el otro.

Se procede a eliminar el duplicado con incorrecto valor ISBN, comprobando previamente que solo sea uno de los registros.

Se ve que los problemas de codificación se ven reflejados en el df final, muchos de los títulos no coinciden por presencia de /, ; y en más de un préstamo, los valores del ISBN (tiisxn) se han escrito con nombres.

Se procede a ver cuántos de estos registros se ven afectados.

In [ ]:
# Se eliminarán aquellos registros que no presenten números en el campo tiisxn
# De no haberlos en el grupo de registros duplicados, se eliminará el segundo.

tiene_num = df_prestamos['tiisxn'].str.contains(r'\d', na=False)

print(f"Registros iniciales: {len(df_prestamos)}")

df_prestamos = (df_prestamos
                .assign(_n=tiene_num)
                .sort_values('_n', ascending=False)
                .drop_duplicates(subset=clave, keep='first')
                .drop(columns='_n')
                .sort_index())

print(f"Registros finales: {len(df_prestamos)}")


del tiene_num
gc.collect()

Registros iniciales: 18254863
Registros finales: 18252727


12199

In [ ]:
df_prestamos.shape

(18252727, 13)

## Ver valores únicos en campos category

A revisar,

prprsu debería tener 64 valores únicos, los mismos que los de la tabla de dimensión. Se eliminan registros que no comiencen por bc o bp que es la nomenclatura de inicio de los códigos prprsu.

prcolp tiene 19 referencias en tabla dimensiones. LEC y LECI predominan, bien son los códigos destinados a lectoras. Ver resto de campos de estos registros para comprobar que son libros

prcocp - me quedo con lo que son prestables-libros, código prl, que suponen 18.228.722. El resto se eliminan.

prlesu - Tiene muchos más valores únicos que prprsu, pero también pueden aparecer los valores de códigos pertenecientes a no bibliotecas públicas. -> se acaba eliminando campo, se toma como referencia el prlebi (códigos de red de bibliotecas).

prcoc1 se ha colado un código ISBN

In [ ]:
for col in df_prestamos.select_dtypes('category').columns:
    print(f"\n{col}  ({df_prestamos[col].nunique(dropna=False)} valores distintos)")
    display(df_prestamos[col].value_counts(dropna=False))


prprsu  (218 valores distintos)


,count
prprsu,
bp16,993620
bp06,862222
bp08,853371
bp05,825856
bp04,790950
...,...
bb01,1
bmcub1,1
bmcad1,1



prcolp  (17 valores distintos)


,count
prcolp,
lec,14744670
leci,3364844
tel,40190
alu,29115
ent,27868
pro,19958
rob,18560
inv,4198
dig,982



prcocp  (9 valores distintos)


,count
prcocp,
prl,18228722
prd,9918
mam,5945
prp,3917
mad,2750
sco,1288
ott,181
ebr,3
jme,3


... (6 outputs left out)


prcocs  (1 valores distintos)


,count
prcocs,
lib,18252727



prcoc1  (8 valores distintos)


,count
prcoc1,
NaN,15184365
adu,1508808
inf,892571
juv,346384
ple,320537
otc,58
xxx,3
978-84-344-3328-1,1


### Código ISBN en campo prcoc1.

Al registro le falta el código. Se le pone en tiisxn y se quita de prcoc1

In [ ]:
# Asignar valor a tiisxn
isbn_encontrado = df_prestamos['prcoc1'] == '978-84-344-3328-1'

df_prestamos.loc[isbn_encontrado, 'tiisxn'] = '978-84-344-3328-1'

In [ ]:
# Quitar categoría de prcoc1
df_prestamos['prcoc1'] = df_prestamos['prcoc1'].cat.remove_categories(['978-84-344-3328-1'])


In [ ]:
len(df_prestamos['prcoc1'].cat.categories)

6

### Eliminar registros no PRL

In [ ]:
print(f"Registros iniciales: {len(df_prestamos)}")
df_prestamos = df_prestamos.loc[df_prestamos['prcocp'] == "prl"]
print(f"Registros finales: {len(df_prestamos)}")

Registros iniciales: 18252727
Registros iniciales: 18228722


### Valores no coincidentes tabla dimensión prprsu

Los códigos de las bibliotecas públicas de Madrid empiezan por BP o BCD seguidas o por siglas del nombre o números, sin espacios entre los caracteres.

Se han encontrado 64.986 registros que no cumplen con el patrón de códigos de las bibliotecas públicas.

Los caracteres encontrados comienzan por BM, que referencia a las Bibliotecas Municipales, una red distinta a la analizada.

Se eliminan estos registros al no haberse realizado los préstamos en los centros de la red observada (BPM).


In [ ]:
# Se buscan los valores que no empiecen por bp o bcd
# y/ o que tengan espacios entre caracteres.
s = df_prestamos['prprsu'].astype('string')

valido = s.str.fullmatch(r'(?:bp|bcd)\S*', case=False, na=False)
no_cumplen = df_prestamos.loc[~valido]

print(f"No cumplen: {len(no_cumplen)} de {len(df_prestamos)}")
display(no_cumplen['prprsu'].value_counts(dropna=False).head(30))

No cumplen: 64986 de 18228722


,count
prprsu,
bmcos1,3464
bmboa2,2924
bmmrz1,2495
bmmaj1,2170
bmsfh1,1877
bmtre1,1776
bmalr1,1745
bmrvm4,1527
bmpaj1,1507


In [ ]:
# Se eliminan registros de los préstamos realizados en red distinta

print(f"Registros iniciales: {len(df_prestamos)}")

df_prestamos = df_prestamos.loc[valido]

print(f"Registros finales: {len(df_prestamos)}")

Registros iniciales: 18228722
Registros finales: 18163736


## Limpieza de campos texto título, autor de patrones no alfabéticos e ISBN no numéricos

Antes de continuar analizando duplicidades por clave se limpia de valores incorrectamente normalizados de las columnas título (tititu), autor (tiauto) e ISBN (tiisxn). Para el título solo se dejarán los componentes alfabéticos, eliminando símbolos, para el autor se eliminarán las fechas y patrones no coincidentes con apellido, autor y para el ISBN, se dejarán los valores numéricos.

### Limpieza texto título

In [ ]:
# Se miran que símbolos no alfanuméricos hay, excluyendo la coma:
titulo= df_prestamos['tititu']
simbolos = titulo.str.findall(r'[^a-zA-Z0-9\s]').explode().value_counts()
print(simbolos.index)

# Se busca qué filas con valores en título:
con_simbolos = titulo.str.contains(r'[^a-zA-Z0-9\s]', na=False)

reg_simbolos= df_prestamos.loc[con_simbolos, 'tititu']
print(len(reg_simbolos))
display(reg_simbolos.head(5))

Index(['/', ':', ',', '!', '¡', '.', '-', '¿', '?', '(', ')', '_', ''', '=',
       '[', ']', ';', '&', '*', 'º', '+', '%', '#', 'ª', '´', '@', '$', '{',
       '|', '`', '·', '®', '²', 'ø', '°', '>', 'æ', 'ð', 'ß', '¹', '­', '^',
       '~', '<', '£', '\', '©', 'þ'],
      dtype='object', name='tititu')
14314467


,tititu
6,"huye rapido, vete lejos"
8,mafalda : tiras de quino
13,"juliet, desnuda"
21,knock out : tres historias de boxeo
23,logicomix : una busqueda epica de la verdad ap...


Se ha encontrado una larga lista tanto de símbolos de puntuación como de valores no normalizados en la lectura, encontrándose en 14.314.467 registros.


['/', ':', ',', '!', '¡', '.', '-', '¿', '?', '(', ')', '_', "'",
       '=', '[', ']', ';', '&', '*', 'º', '+', '%', '#', 'ª', '´', '@',
       '$', '{', '|', '`', '·', '®', '²', 'ø', '°', 'æ', '>', 'ð', 'ß',
       '¹', '\xad', '^', '~', '<', '£', '\\', '©', 'þ']



Se proceden a eliminar de los registros. Distinguiendo entre los que no dejan espacio (xad·'´`ºª°¹²®©) y los que sí (el resto).
Los signos de puntuación se eliminan con tal de dejar el texto lo más limpio posible y así poder cruzarlo entre sí y con tabla registros, evitando posibles diferencias derivadas de una incorrecta transcripción.

Se mantienen los números y letras extranjeras.

In [ ]:
# Eliminación símbolos y signos de puntuación:

sin_espacio = r"[\xad·'´`ºª°¹²®©]"

df_prestamos['tititu'] = (
    df_prestamos['tititu']
    .str.replace(sin_espacio, '', regex=True)
    .str.replace(r'[^\w\s]|[\d_]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Comprobación
df_prestamos['tititu'].str.contains(r'[^a-zA-Z0-9\s]', na=False).sum()


np.int64(200)

### Limpieza texto autor

Se eliminan los años de nacimiento y muerte, junto con paréntesis y "-". Se mantiene la coma que separa el apellido del nombre y letras extranjeras.

In [ ]:
# Se miran que símbolos no alfanuméricos hay, excluyendo la coma:
autor= df_prestamos['tiauto']
numeros = autor.str.findall(r'[^a-zA-Z0-9\s,]').explode().value_counts()
print(numeros.index)

# Se busca qué filas con valores en autor:
con_numeros = autor.str.contains(r'[^a-zA-Z0-9\s,]', na=False)

reg_numeros= df_prestamos.loc[con_numeros, 'tiauto']
print(len(reg_numeros))
display(reg_numeros.head(5))

Index(['-', '(', ')', '.', ''', '/', 'ø', '&', '?', '[', '¿', '@', '#', ':',
       ']', ';', 'ª', '¡', 'ð', '!', '=', 'º', '_', '|', '´', '$', 'ß', '·',
       '+', '¥', '…', '’', 'þ', 'æ', '`', '<', '>'],
      dtype='object', name='tiauto')
12717866


,tiauto
0,"obiols, miquel (1945-)"
1,"rodari, gianni (1920-1980)"
3,"asensi, matilde (1962-)"
4,"salinger, j. d. (1919-2010)"
5,"munoz martin, juan (1929-)"


Se han encontrado los siguientes símbolos no alfanuméricos en el campo autor.

['-', '(', ')', '.', ''', '/', 'ø', '&', '?', '[', '¿', '@', '#', ':',
       ']', ';', 'ª', '¡', 'ð', '!', '=', 'º', '_', '|', '´', '$', 'ß', '·',
       '+', '¥', '…', '’', 'þ', 'æ', '`', '<', '>']

Repartidos en 12.717.866 registros. Se eliminan todos. Se procede a eliminar de los registros. Distinguiendo entre los que no dejan espacio (xad·'´`ºª°¹²®©) y los que sí (el resto).

In [ ]:
# Eliminación caracteres y números años de nacimiento y muerte

sin_espacio = r"[\xad·'´`ºª°¹²®©]"

df_prestamos['tiauto'] = (
    df_prestamos['tiauto']
    .str.replace(r'\s*\([^)]*\)', '', regex=True)
    .str.replace(sin_espacio, '', regex=True)
    .str.replace(r'[^\w\s,]|[\d_]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(r'\s*,\s*', ', ', regex=True)
    .str.replace(r',(\s*,)+', ',', regex=True)
    .str.strip(' ,'))

# Comprobación

df_prestamos['tiauto'].str.contains(r'[^a-zA-Z0-9\s,]', na=False).sum()

np.int64(24053)

### Limpieza texto ISBN

Primero se pasan como nulo los valores de los registros que NO tienen números.

Después se revisan, los posibles símbolos que pueda haber y se eliminan. Especialmente se extraen los "-" que separan grupos de números para dejar los valores lo más normalizados posibles.

In [ ]:
# Se miran que registros no son nulos ni contienen números:
# Se trabaja sobre serie:

isbn = df_prestamos['tiisxn']

mask_isbn = isbn.notna() & ~isbn.str.contains(r'[0-9]', na=False)

sin_numeros_isbn = df_prestamos.loc[mask_isbn, 'tiisxn']
print(len(sin_numeros_isbn))
sin_numeros_isbn.head(10)

22193


,tiisxn
2671727,cancione
2672015,las aves
2672021,hipolito /
2672568,"more, elizabeth ann"
2672860,"mate, ferenc"
2672940,lengua silenciosa /
2673326,malvaloca /
2673385,romp
2673839,los jugadores /
2674374,el jardin de los


In [ ]:
del sin_numeros_isbn

Se han encontrado 22.193 registros con valores alfabéticos en el campo ISBN por la incorrecta decodificación de los CSV. Se pasan a nulo.

In [ ]:
# Se pasan estos registros a nulos por incorrecta decodificación:

print(f"Nulos campos iniciales {df_prestamos['tiisxn'].isna().sum()}")
df_prestamos.loc[mask_isbn, 'tiisxn'] = pd.NA
print(f"Nulos campos finales {df_prestamos['tiisxn'].isna().sum()}")

Nulos campos iniciales 3213375
Nulos campos finales 3235568


In [ ]:
# Se buscan símbolos no numéricos en isbn:

no_numeros = isbn.str.findall(r'[^0-9\s]').explode().value_counts()
print(no_numeros.reset_index()['tiisxn'].unique())

# Se busca qué filas con valores en autor:
mask_isbn = isbn.str.contains(r'[^0-9\s]', na=False)

reg_numeros_isbn = df_prestamos.loc[mask_isbn, 'tiisxn']
print(len(reg_numeros_isbn))
display(reg_numeros_isbn.head(5))

['-' '(' 'e' ',' 'x' '+' ':' 'a' 'o' 'i' 'r' 'n' 'l' 's' ')' 't' 'c' 'd'
 'u' 'm' 'g' 'b' 'p' 'h' 'v' '.' 'j' 'k' 'z' 'y' 'f' '/' 'w' 'q' '[' '©'
 '¡' ']' '³' "'" '\xad' '#' '&' '!' '"' '_' '?' '¿' '¯' '±' 'º' 'ø' '¨'
 '>' 'ª' '‰' '=' '\x81' '²' ';' '¼' '¶' '§' '$' '\x8d' '¤' '«']
14895355


,tiisxn
2671144,84-9734-142-2 :
2671145,84-350-1426-6
2671146,978-84-9800-404-5
2671147,978-84-9066-319-6
2671148,84-95178-38-9


Se han encontrado los siguientes símbolos: ['-' '(' 'e' ',' 'x' '+' ':' 'a' 'o' 'i' 'r' 'n' 'l' 's' ')' 't' 'c' 'd'
 'u' 'm' 'g' 'b' 'p' 'h' 'v' '.' 'j' 'k' 'z' 'y' 'f' '/' 'w' 'q' '[' '©'
 '¡' ']' '³' "'" '\xad' '#' '&' '!' '"' '_' '?' '¿' '¯' '±' 'º' 'ø' '¨'
 '>' 'ª' '‰' '=' '\x81' '²' ';' '¼' '¶' '§' '$' '\x8d' '¤' '«']

En 14.959.101 registros.
Se observan fallos de decodificación, se vuelven a decodificar.
Se eliminan y no se dejan espacios en su lugar.

In [ ]:
# Decodificación correcta:

# Número y visualización de los valores:
mask_isbn_deco = isbn.str.contains(r'[^0-9\s]', na=False)
print(mask_isbn_deco.sum())
display(isbn[mask_isbn_deco].head(5))

#Decodificación

def reparar(t):
    try:
        return t.encode('latin-1').decode('cp1252')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return t

df_prestamos.loc[mask_isbn_deco, 'tiisxn'] = isbn[mask_isbn_deco].map(reparar)

#Comprobación
df_prestamos['tiisxn'].str.contains(r'[\x80-\x9f]', na=False).sum()

14895355


,tiisxn
2671144,84-9734-142-2 :
2671145,84-350-1426-6
2671146,978-84-9800-404-5
2671147,978-84-9066-319-6
2671148,84-95178-38-9


np.int64(13)

In [ ]:
# Eliminación símbolos y signos de puntuación:

df_prestamos['tiisxn'] = (
    df_prestamos['tiisxn']
    .str.replace(r'[^0-9\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print(df_prestamos['tiisxn'].str.contains(r'[^0-9\s]', na=False).sum())

0


,tiisxn
13547922,9788420439389
14344860,9781405884457
13750157,9788494603525
8255937,9788494247330
9645116,9788418395109


Según normas y estandarización del ISBN consta de 13 dígitos desde 2007, antes era de 10 dígitos.

Se comprueba longitud de los ISBN resultantes.

In [ ]:
# Longitud de los códigos :

df_prestamos['tiisxn_longitud'] = df_prestamos['tiisxn'].str.len()

display(df_prestamos['tiisxn_longitud'].value_counts(normalize=True, dropna= False).sort_index()*100)


,proportion
tiisxn_longitud,
0.0,0.122183
1.0,0.000006
2.0,0.001283
3.0,0.006210
4.0,0.082791
5.0,0.089172
6.0,0.001151
7.0,0.055005
8.0,0.908068


Hay gran dispersión de longitudes en códigos ISBN, pero el 80% de los registros tiene ISBN con longitudes aptas. Un 17,7 % de valores nulos.

Más adelante se observa cantidad de Ejemplares por código ISBN y se comprueba que un ejemplar tiene un único código.

## Se vuelven a mirar duplicados completos después de la normalización.

In [ ]:
# Se mira cantidad de grupos de registros completamente duplicados:

duplis_completos = df_prestamos[df_prestamos.duplicated(keep=False)]

tam = duplis_completos.value_counts(dropna=False) # Duplicados teniendo en cuenta nulos
n_grupos = len(tam)

print(f"Numero de grupos registros duplicados: {len(tam)}\n"
f"\nGrupos de duplicados con más de dos registros {len(tam[tam > 2])}")

ej_eliminar = tam.sum() - len(tam)

print(f"\nSe eliminaran {ej_eliminar} registros."
f"Dejando {len(df_prestamos) - ej_eliminar} registros restantes")

Numero de grupos registros duplicados: 0

Grupos de duplicados con más de dos registros 0

Se eliminaran 0 registros.Dejando 18163736 registros restantes


## Duplicados por clave prbar + prprsu + prfpre (Código ejemplar + biblioteca emisora + fecha)

Para detectar duplicidades por biblioteca y día. Asumiendo la idea de que el préstamo de un mismo ejemplar no ocurre el mismo día. Sin posibilidad real de discernir al usuario. Más allá de la anteriormente expuesta.

In [ ]:
# Duplicados en clave:
clave = ['prbarc', 'prprsu', 'prfpre']

duplis_clave2 = df_prestamos[df_prestamos.duplicated(subset=clave, keep=False)]

if duplis_clave2.empty:
    print("No hay registros duplicados según la clave")
else:
    tam3 = duplis_clave2.value_counts(subset=clave, dropna=False)
    print(f"Grupos duplicados: {len(tam3)}")
    print(f"Registros implicados: {len(duplis_clave2)}")
    print(f"Registros sobrantes (si dejas 1 por grupo): {len(duplis_clave2) - len(tam3)}")
    print("\nDistribución de tamaños de grupo:")
    print(tam3.value_counts().sort_index())
    print(tam3.value_counts(normalize=True).sort_index())
    display(tam3.head(10))

Grupos duplicados: 35593
Registros implicados: 71569
Registros sobrantes (si dejas 1 por grupo): 35976

Distribución de tamaños de grupo:
count
2    35269
3      273
4       43
5        8
Name: count, dtype: int64
count
2    0.990897
3    0.007670
4    0.001208
5    0.000225
Name: proportion, dtype: float64


,,,count
prbarc,prprsu,prfpre,
100925948,bcdij,2020-03-04,5
101340002,bcdij,2019-10-02,5
101452646,bcdij,2019-10-02,5
7143422,bp11,2018-08-14,5
1400387765,bcdij,2019-10-02,5
1400034896,bcdij,2019-10-02,5
1400241770,bcdij,2019-10-02,5
1400955874,bcdij,2019-10-02,5
1400972864,bcdij,2020-10-23,4


Se han encontrado 71.569 registros que comparten la clave - ['prbarc', 'prprsu', 'prfpre'].
Repartidos en 35.593 grupos
Grupos de duplicados con más de dos registros 324

Se eliminarán 35.976 registros.

Se observa que la casuística es poco común apenas un 0,20% de los registros observados cumple esta duplicidad.

El 99% de los grupos encontrados tiene solo dos registros duplicados y como máximo, se han encontrado grupos de hasta 5 registros duplicados.

Se procede a buscar las distintas causas por las que aparecen las diferencias. Si es por título, autor, isbn incorrectamente decodificados o si realmente se han realizado préstamos del mismo ejemplar con carnets distintos (prlesu -> biblioteca donde se hizo el carnet, pradul -> si carnet adulto o infantil).

In [ ]:
clave = ['prbarc', 'prprsu', 'prfpre']

# Se trabaja sobre el df con solo los duplicados encontrados por clave
duplis_clave2 = duplis_clave2[duplis_clave2.duplicated(subset=clave, keep=False)]

# Asignación de un id por grupo.
d = (duplis_clave2
     .assign(grupo=lambda x: x.groupby(clave, dropna=False).ngroup()) # Se tiene en cuenta nulos en clave
     .sort_values(clave)) # Que si los hay. 4 valores en prprsu

# Qué columnas varían
otras = [c for c in d.columns if c not in clave + ['grupo']]
difiere = d.groupby('grupo')[otras].nunique(dropna=False).gt(1)
cols_dif = difiere.columns[difiere.any()].tolist()

print(f"Claves con diferencias: {len(difiere)}")
display(difiere[cols_dif].sum().sort_values(ascending=False).to_frame('n_claves'))

# 3. Combinaciones de columnas que cambian juntas
patron = difiere[cols_dif].apply(lambda r: ', '.join(r.index[r]), axis=1)
display(patron.value_counts().to_frame('n_claves'))

# 4. Ejemplos por columna (grupos completos, ordenados)
for col in cols_dif:
    grupos = difiere.index[difiere[col]][:5]          # 5 grupos, no 10 filas
    print(f"\n{col}: {difiere[col].sum()} claves")
    display(d.loc[d['grupo'].isin(grupos), clave + [col]])

/tmp/ipykernel_823/3002392239.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .assign(grupo=lambda x: x.groupby(clave, dropna=False).ngroup()) # Se tiene en cuenta nulos en clave


Claves con diferencias: 35593


,n_claves
prlesu,26570
pradul,14507
prcolp,11580
prlebi,5898
tiisxn,5
tiisxn_longitud,5


,n_claves
prlesu,13719
"prcolp, pradul",3931
"prlebi, prlesu",3861
"prcolp, prlesu, pradul",3300
pradul,3174
"prlesu, pradul",2493
prcolp,1911
"prcolp, prlesu",1163
"prcolp, prlebi, prlesu, pradul",843
"prlebi, prlesu, pradul",759



prcolp: 11580 claves


,prbarc,prprsu,prfpre,prcolp
5575022,100003268,bcdin,2021-02-01,leci
5576070,100003268,bcdin,2021-02-01,lec
11262086,100004076,bcdii,2023-12-28,lec
11286369,100004076,bcdii,2023-12-28,leci
2568886,100009267,bcdia,2019-03-20,lec
2569279,100009267,bcdia,2019-03-20,leci
10691391,100017043,bcdib,2023-09-27,rob
10743782,100017043,bcdib,2023-09-27,lec
2071814,100036240,bcdiu,2019-01-31,leci
2257441,100036240,bcdiu,2019-01-31,lec


... (6 outputs left out)


tiisxn: 5 claves


,prbarc,prprsu,prfpre,tiisxn
14969519,1402487445,bcdis,2025-04-01,97884112
15008030,1402487445,bcdis,2025-04-01,9788413921143
14965936,1402487566,bcdi8,2025-04-29,97827312
15007994,1402487566,bcdi8,2025-04-29,9782733899953
14975002,2000269974,bp04,2025-04-13,97884512
15008106,2000269974,bp04,2025-04-13,9788448867225
14935150,2000534525,bp04,2025-04-03,97884212
15007659,2000534525,bp04,2025-04-03,9788417424596
14984133,2001378121,bp21,2025-04-23,97884112
15008178,2001378121,bp21,2025-04-23,9788411312516



tiisxn_longitud: 5 claves


,prbarc,prprsu,prfpre,tiisxn_longitud
14969519,1402487445,bcdis,2025-04-01,8.0
15008030,1402487445,bcdis,2025-04-01,13.0
14965936,1402487566,bcdi8,2025-04-29,8.0
15007994,1402487566,bcdi8,2025-04-29,13.0
14975002,2000269974,bp04,2025-04-13,8.0
15008106,2000269974,bp04,2025-04-13,13.0
14935150,2000534525,bp04,2025-04-03,8.0
15007659,2000534525,bp04,2025-04-03,13.0
14984133,2001378121,bp21,2025-04-23,8.0
15008178,2001378121,bp21,2025-04-23,13.0


Se eliminan los registros con isbn len 8

Hay duplicidades causadas por retirada con tipo carnet adulto/infantil -> se deja el infantil.
Misma casuística para los tipos de lector, entre lec (lector adulto) y leci (lector infantil), se mantienen los infantiles. Si otra casuística se mantiene el código lec

Duplicidades por origen de creación del carnet de retirada distinto, tanto para el código específico de la biblioteca (prlesu) como para el de red de bibliotecas (prlebi). Solo se eliminan los registros nulos, en el resto de los casos no se puede determinar si es préstamo real o duplicado, se deja como real. -> Se añade a clave

In [ ]:
clave_ant = ['prbarc', 'prprsu', 'prfpre']
clave = clave_ant + ['prlesu']

def aplicar_regla(df, nombre, puntua, clave):
    valor = puntua(df)
    mejor = valor.groupby([df[c] for c in clave], observed=True, dropna=False).transform('min')
    eliminar = valor > mejor
    print(f"{nombre}: {eliminar.sum()} eliminados")
    return df.loc[~eliminar]

rank_colp = lambda d: d['prcolp'].astype('string').map({'leci': 0, 'lec': 1}).fillna(2)

n0 = 0

print(f"Duplicados exactos: {n0 - len(df_prestamos)}")

# Se quitan los nulos en los duplicados de prlesu con la clave ANTIGUA
df_prestamos = aplicar_regla(df_prestamos, "prlesu nulo",
                             lambda d: d['prlesu'].isna().astype(int), clave_ant)

# Se aplican las reglas: con la clave actualizada
reglas = [
    ("tiisxn longitud 8", lambda d: d['tiisxn_longitud'].eq(8).astype(int)),
    ("pradul: gana 0",    lambda d: pd.to_numeric(d['pradul'], errors='coerce')),
    ("prcolp: leci > lec > otros", rank_colp),
]
for nombre, puntua in reglas:
    df_prestamos = aplicar_regla(df_prestamos, nombre, puntua, clave)

print(f"Registros: {n0} -> {len(df_prestamos)}")

Duplicados exactos: -18163736
prlesu nulo: 795 eliminados
tiisxn longitud 8: 2 eliminados
pradul: gana 0: 7197 eliminados
prcolp: leci > lec > otros: 1882 eliminados
Registros: 0 -> 18153860


## Se observan valores nulos

In [ ]:
# Se observan valores nulos en campos:
df_prestamos.isna().sum()

,0
prbarc,0
prprsu,0
prcolp,32
prcocp,0
prlebi,32
prlesu,130021
pradul,32
prfpre,0
prcocs,0
tititu,65


Se eliminan los registros donde no se muestre o prlebi o prlesu, ya que es la base por la que distinguir al usuario que realiza el préstamo.

Los nulos prlesu, no se pueden asignar, se dejan como están.

Para los nulos de tititu, tiauto y tiisxn (valorando que tengan longitud correcta 13 o 10 dígitos), se busca asignar mediante otros registros con el mismo código de barras (prbarc). Si no está en la tabla se mira si se puede conseguir vía tabla registros.

Para los nulos de prcoc1, que empezó a aparecer en julio'25, se busca asignar por código de barras y para los restantes se mira asignar vía tabla de registros con el campo tipo_publico sea adaptable.


### Nulos prlebi + prcolp + pradul

Coinciden los registros nulos para los campos prlebi, prcolp y pradul.

Se eliminan de la tabla central.

In [ ]:
nulos_prelebi = df_prestamos['prlebi'].isna()

print(f"Registros iniciales: {len(df_prestamos)}")

# Se guarda df sin estos valores

df_prestamos = df_prestamos.loc[~nulos_prelebi]



In [ ]:
print(f"Registros finales: {len(df_prestamos)}")

Rgistros iniciales: 18153828


In [ ]:
df_prestamos.shape

(18153828, 14)

### Nulos título

In [ ]:
nulos_titulo = df_prestamos['tititu'].isna()

# Se extraen los prbarc de los registros con títulos nulos

# DF de títulos nulos:
titulo_nulos = df_prestamos.loc[nulos_titulo]

# Prbarc únicos en títulos nulos:
prbar_titulos_nulos = titulo_nulos['prbarc'].unique()
print(f"Codigos unicos en titulo_nulos {len(prbar_titulos_nulos)}")
print(prbar_titulos_nulos)

Codigos unicos en titulo_nulos 30
['2266199' '7708943' '2159871' '1400558971' '7121876' '9142933'
 '1400307352' '1400475315' '5192225' '7699826' '1400728978' '7556941'
 '1400398004' '1400297161' '1400120790' '100084605' '1400465996' '1934727'
 '9306616' '7249449' '2000020891' '2105323' '7663634' '100479482'
 '1400964537' '7250773' '9371047' '7803851' '2000095517' '1400851085']


Se han encontrado 30 códigos únicos en los 65 registros con título nulo.

['2266199' '7708943' '2159871' '1400558971' '7121876' '9142933'
 '1400307352' '1400475315' '5192225' '7699826' '1400728978' '7556941'
 '1400398004' '1400297161' '1400120790' '100084605' '1400465996' '1934727'
 '9306616' '7249449' '2000020891' '2105323' '7663634' '100479482'
 '1400964537' '7250773' '9371047' '7803851' '2000095517' '1400851085']

Se extrae diccionario según código de barras para asignar título si coincidencia. Para ello se analiza si valores únicos y coincidentes entre tititu y prbarc.

In [ ]:
# Se ve si hay más de un título por código de barras del ejemplar.

conflictos = df_prestamos.groupby('prbarc')['tititu'].nunique()
conflictos[conflictos > 1]

,tititu
prbarc,
100001273,2
100001377,2
100001382,2
100001385,2
100001420,2
...,...
9539574,2
9539584,2
9539723,2


In [ ]:
# prbarc con más de un título distinto
n_titulos = df_prestamos.groupby('prbarc')['tititu'].nunique()
prbarc_conflicto = n_titulos[n_titulos > 1].index
print(f"prbarc en conflicto: {len(prbarc_conflicto)}")

# Detalle: cada título y cuántas veces aparece por prbarc
detalle = (df_prestamos[df_prestamos['prbarc'].isin(prbarc_conflicto)]
           .groupby(['prbarc', 'tititu'])
           .size()
           .reset_index(name='n')
           .sort_values(['prbarc', 'n'], ascending=[True, False]))
detalle['n_titulos'] = detalle.groupby('prbarc')['tititu'].transform('nunique')
detalle['pct'] = detalle['n'] / detalle.groupby('prbarc')['n'].transform('sum')
detalle.head(5)

prbarc en conflicto: 74724


,prbarc,tititu,n,n_titulos,pct
0,100001273,la venus de cobre la iii novela de marco didio...,1,2,0.50
1,100001273,la venus de cobre una novela de marco didio falco,1,2,0.50
2,100001377,words pictures diccionario en imagenes ingles,3,2,0.75
3,100001377,words pictures diccionario en imagenes para ni...,1,2,0.25
4,100001382,words pictures diccionario en imagenes ingles,1,2,0.50


Al ver si hay un título único por prbarc, se ve que no es así. Hay prbarc a lo largo de la tabla que varían el título en detalles.

Para igualar todos los títulos, en caso de diferencias, se toma el título que aparece más prestado, empleando la moda, y se asigna a los préstamos de su prbarc.


In [ ]:
# Se busca título a asignar

base = df_prestamos.dropna(subset=['tititu']).reset_index(drop=True)
base['pos'] = base.index   # posición en el orden original del DataFrame

conteo = (base.groupby(['prbarc', 'tititu'], sort=False)
          .agg(n=('pos', 'size'), primera_pos=('pos', 'min'))
          .reset_index())
conteo['pct'] = conteo['n'] / conteo.groupby('prbarc')['n'].transform('sum')

ganador = (conteo
           .sort_values(['prbarc', 'pct', 'primera_pos'], ascending=[True, False, True])
           .drop_duplicates('prbarc', keep='first')
           .set_index('prbarc')['tititu'])

In [ ]:
# Se asigna título a todos los registros con mismo prbarc

df_prestamos['tititu_imp'] = df_prestamos['prbarc'].map(ganador).fillna(df_prestamos['tititu'])

In [ ]:
# Se crea nueva columna para trazabilidad de las asignaciones

df_prestamos['origen_titulo'] = np.select(
    [df_prestamos['tititu'].isna() & df_prestamos['tititu_imp'].notna(),
     df_prestamos['tititu'].notna() & (df_prestamos['tititu'] != df_prestamos['tititu_imp']),
     df_prestamos['tititu_imp'].isna()],
    ['imputado', 'corregido', 'sin_resolver'],
    default='original'
)
df_prestamos['origen_titulo'].value_counts()

,count
origen_titulo,
original,17930942
corregido,222821
imputado,57
sin_resolver,8


In [ ]:
nulos = df_prestamos[df_prestamos['tititu_imp'].isna()]

print("prbarc nulo:", nulos['prbarc'].isna().sum())
print("código sin ningún título informado:", (~nulos['prbarc'].isin(conteo['prbarc']) & nulos['prbarc'].notna()).sum())

prbarc nulo: 0
código sin ningún título informado: 8
código sin ganador: 0


Quedan 8 títulos sin ningún título informado, códigos sin información. Se revisa si códigos en tabla ejemplares y si registros asociados en tabla registros. Se extrae lista de códigos únicos sin título.

In [ ]:
df_prestamos[['tititu', 'tititu_imp', 'origen_titulo']].isna().sum()

,0
tititu,65
tititu_imp,8
origen_titulo,0


In [ ]:
# Códigos sin título asociado:
nulos_titulo = df_prestamos['tititu_imp'].isna()

# Se extraen los prbarc de los registros con títulos nulos

# DF de títulos nulos:
titulo_nulos = df_prestamos.loc[nulos_titulo]

# Prbarc únicos en títulos nulos:
prbar_titulos_nulos = titulo_nulos['prbarc'].unique()
print(f"Codigos unicos en titulo_nulos {len(prbar_titulos_nulos)}")
print(prbar_titulos_nulos)

Codigos unicos en titulo_nulos 4
['1400307352' '1400475315' '100084605' '100479482']


Los 8 registros sin título, van desde el año 2018 al 2023. Presentan o autor declarado (6 de los 8) o isbn válido (4 de 8). Se genera lista de los códigos implicados para ver si está en la tabla de ejemplares y así poder extraer el título desde la tabla de registros.

In [ ]:
lista_titulos_nulos = list(prbar_titulos_nulos)


In [ ]:
# Se mira si prbarc de los títulos nulos en tabla ejemplares:

 # Se solicitan las columnas barcode y id_registro para consultar
 # tabla registros de haber coincidencia.

ejemplares_codigos = pd.read_parquet(
    ORIGEN_LOCAL/'ejemplares.parquet',
    engine="pyarrow",
    columns=['barcode_original', 'id_registro'],
    filters=[('barcode_original', 'in', lista_titulos_nulos)])

In [ ]:
# Se mira resultado:

# Se observa que códigos se han encontrado y cuales no:
encontrados = set(ejemplares_codigos['barcode_original'])
no_encontrados = set(lista_titulos_nulos) - encontrados
print(f" Se han encontrado {len(encontrados)} códigos barcode_original de los "
f"{len(lista_titulos_nulos)} códigos prbarc solicitados.\n"
f" No hay coincidencia para {len(no_encontrados)} códigos prbarc solicitados.")

 Se han encontrado 4 códigos barcode_original de los 4 códigos prbarc solicitados.
 No hay coincidencia para 0 códigos prbarc solicitados.


In [ ]:
# Se observan valores únicos para id_registro
id_registro_unicos = ejemplares_codigos['id_registro'].unique()

barcode_x_registro = ejemplares_codigos.groupby('id_registro')['barcode_original'].unique()

print(f"Hay {len(id_registro_unicos)} para los {len(encontrados)}.\n"
f"Siguiendo la siguiente distribucion: \n\n {barcode_x_registro}")

Hay 3 para los 4.
Siguiendo la siguiente distribucion: 

 id_registro
es-marbcm00175424c-2    [100084605, 100479482]
es-marbcm00400801c-x              [1400475315]
es-marbcm01759241c-1              [1400307352]
Name: barcode_original, dtype: object


In [ ]:
# Se extrae título de la tabla registros:

titulos_registros = pd.read_parquet(
    ORIGEN_LOCAL/'registros1_1.parquet',
    engine="pyarrow",
    columns=['id_registro', 'titul_norm'],
    filters=[('id_registro', 'in', list(id_registro_unicos))])

In [ ]:
# Se añade barcode por id_registro para imputar según el:

titulos_registros = titulos_registros.merge(ejemplares_codigos, on='id_registro')

In [ ]:

# Verificación de un título por barcode/prbarc

titulos_duplicados = titulos_registros['barcode_original'].duplicated(keep=False)
if titulos_duplicados.any():
    print("Barcodes con más de un título:")
    print(titulos_registros.loc[titulos_duplicados].sort_values('barcode_original'))
else:
  print("No hay títulos duplicados por código de ejemplar")


No hay títulos duplicados por código de ejemplar


In [ ]:
# Asignación del título al prbarc de la tabla prestamos

# Se asigna diccionario con k-v como código ejemplar - título

dict_titulos = (
    titulos_registros
    .drop_duplicates(subset='barcode_original')
    .set_index('barcode_original')['titul_norm']
    .to_dict()
)

# Asignación valor a los registros con títulos nulos:

df_prestamos['tititu_imp'] = df_prestamos['tititu_imp'].fillna(
    df_prestamos['prbarc'].map(dict_titulos)
)

# Comprobaciones:

    # Títulos no nulos finales, si coincidencia len igual a nulos_titulo:
titulos_imputados = nulos_titulo & df_prestamos['tititu_imp'].notna()
print(f"Titulos nulos iniciales en columna tititu_imp de la tabla prestamos: "
f"{len(titulo_nulos)}.")
print(f"Titulos imputados: {(titulos_imputados.sum())}")
print(f"Titulos nulos finales en columna tititu_imp: "
f"{df_prestamos['tititu_imp'].isna().sum()}.")

# Registro origen imputación:
df_prestamos.loc[nulos_titulo, 'origen_titulo'] = 'tabla registros'

Titulos nulos iniciales en columna tititu_impt de la tabla prestamos: 8.
Titulos imputados: 8
Titulos nulos finales en columna tititu_imp: 0.


### Nulos autor

In [ ]:

# Máscara booleana de autores nulos
nulos_autor = df_prestamos['tiauto'].isna()

# Se extraen los prbarc de los registros con autores nulos

# DF de autores nulos:
autor_nulos = df_prestamos.loc[nulos_autor]

# Prbarc únicos en autores nulos:
prbar_autor_nulos = autor_nulos['prbarc'].unique()
print(f"Codigos unicos en autor_nulos {len(prbar_autor_nulos)}")
print(prbar_autor_nulos)

Codigos unicos en autor_nulos 165774
['270007' '1879526' '1879586' ... '2001387589' '2001356979' '2000876289']


Se han encontrado 165.774 códigos prbarc únicos en los 853.384 registros con autor nulo.

Se extrae diccionario según código de barras para asignar autor si coincidencia. Para ello se analiza si valores únicos y coincidentes entre tiauto y prbarc.

In [ ]:
# Se ve si hay más de un autor por código de barras del ejemplar.

conflictos = df_prestamos.groupby('prbarc')['tiauto'].nunique()
prbarc_con = conflictos[conflictos > 1].index
autores_prbar = conflictos.value_counts()
print(f"Hay {len(prbarc_con)} códigos de ejemplar con más de un autor asignado")
print(f"Para cada código de ejemplar se pueden encontrar hasta "
f"{autores_prbar} nombres de autores distintos. ")

Hay 51360 códigos de ejemplar con más de un autor asignado
Para cada código de ejemplar se pueden encontrar hasta tiauto
1    2441825
0     154271
2      49814
3       1424
4         92
5         27
6          3
Name: count, dtype: int64 nombres de autores distintos. 


Encontramos que hay 154.271 códigos de ejemplar que NO tienen ningún autor asociado. Se deberán buscar en registros y se ve que hay 51.360 códigos de ejemplar con más de un autor asociado, se procede a ver qué ha podido suceder.

In [ ]:
# Detalle: cada autor y cuántas veces aparece por prbarc

detalle = (df_prestamos[df_prestamos['prbarc'].isin(prbarc_con)]
           .groupby(['prbarc', 'tiauto'])
           .size()
           .reset_index(name='n')
           .sort_values(['prbarc', 'n'], ascending=[True, False]))
detalle['n_autores'] = detalle.groupby('prbarc')['tiauto'].transform('nunique')
detalle['pct'] = detalle['n'] / detalle.groupby('prbarc')['n'].transform('sum')
detalle.head(5)

,prbarc,tiauto,n,n_autores,pct
1,100001348,"leralta, javier",2,2,0.666667
0,100001348,"leralta garcia, javier",1,2,0.333333
2,100001649,"alberti, rafael",1,2,0.500000
3,100001649,el adefesio,1,2,0.500000
4,100001693,arte de amar,1,2,0.500000


A raíz de la muestra, se observa que una de las causas por las que hay más de un autor asociado a un código de barras es la aparición del apellido completo, mientras que en otros registros solo aparece el primer apellido.

Otra de las causas es que no hay nombre autor en el registro, sino que aparece el título del autor. Se distinguirán estos y se reasignarán por el valor del grupo que presente una coma, entendiendo que esta separa el apellido del nombre. Se unifican nombres en todos los registros de cada grupo.

In [ ]:
# Se busca autor a asignar

# Se establece la base del diccionario de asignación:
# Solo autores informados:

base_autores = df_prestamos.dropna(subset=['tiauto']).reset_index(drop=True)
base_autores['pos'] = base_autores.index

# Rasgos para la prioridad:

base_autores['tiene_coma'] = base_autores['tiauto'].str.contains(',', regex=False)
antes_coma = base_autores['tiauto'].str.split(',', n=1).str[0].str.strip()
base_autores['n_pal'] = antes_coma.str.split().str.len()

base_autores['prioridad'] = np.select(
    [base_autores['tiene_coma'] & (base_autores['n_pal'] >= 2),
     base_autores['tiene_coma']],
    [2, 1],
    default=0
)

# Conteo por grupo
conteo = (base_autores.groupby(['prbarc', 'tiauto'], sort=False)
          .agg(n=('pos', 'size'),
               primera_pos=('pos', 'min'),
               prioridad=('prioridad', 'first'))
          .reset_index())
conteo['pct'] = conteo['n'] / conteo.groupby('prbarc')['n'].transform('sum')

# Ganador por prbarc: prioridad > frecuencia > primera aparición
ganador_df = (conteo
              .sort_values(['prbarc', 'prioridad', 'pct', 'primera_pos'],
                           ascending=[True, False, False, True])
              .drop_duplicates('prbarc', keep='first')
              .set_index('prbarc'))
ganador_autor = ganador_df['tiauto']


In [ ]:
# Se asignan autores confirmados a todos los registros con mismo prbarc

autor_ref = df_prestamos['prbarc'].map(ganador_autor)

prbarc_fiables = ganador_df.index[ganador_df['prioridad'] > 0]
mask_homog = df_prestamos['prbarc'].isin(prbarc_fiables)
df_prestamos['tiauto_imp'] = df_prestamos['tiauto']
df_prestamos.loc[mask_homog, 'tiauto_imp'] = autor_ref[mask_homog]

In [ ]:
# Se crea nueva columna para trazabilidad de las asignaciones

df_prestamos['origen_autor'] = np.select(
    [df_prestamos['tiauto'].isna() & df_prestamos['tiauto_imp'].notna(),
     df_prestamos['tiauto'].notna() & (df_prestamos['tiauto'] != df_prestamos['tiauto_imp']),
     df_prestamos['tiauto_imp'].isna()],
    ['imputado', 'corregido', 'sin_resolver'],
    default='original'
)
df_prestamos['origen_autor'].value_counts()

,count
origen_autor,
original,17094775
sin_resolver,816476
corregido,205671
imputado,36906


In [ ]:
nulos = df_prestamos[df_prestamos['tiauto_imp'].isna()]

prbarc_con_autor = ganador_df.index
prbarc_prio0 = ganador_df.index[ganador_df['prioridad'] == 0]

print("prbarc nulo:", nulos['prbarc'].isna().sum())
print("código sin ningún autor informado:", (~nulos['prbarc'].isin(conteo['prbarc']) & nulos['prbarc'].notna()).sum())
print("código con autores sin coma", nulos['prbarc'].isin(prbarc_prio0).sum())

prbarc nulo: 0
código sin ningún autor informado: 804487
código con autores sin coma 11989


Quedan 813.943 registros que corresponden a 155.749 ejemplares únicos de los 853.384 iniciales sin autor informado. Se revisa si códigos en tabla ejemplares y si registros asociados en tabla registros. Se extrae lista de códigos únicos sin autor.



**Asignar autores nulos mediante info en registros**

In [ ]:
df_prestamos['tiauto_imp'].isna().sum()

np.int64(813943)

In [ ]:
autor_nulos = df_prestamos[df_prestamos['tiauto_imp'].isna()]
autor_nulos['prbarc'].unique()

<ArrowStringArray>
[    '270007',    '1879526',    '1879586',    '1879849',    '1880026',
    '1880232',    '1881417',    '1881608',    '1881759',    '1881764',
 ...
 '1402345940', '2001356948', '2001397164', '2001241153', '2000714527',
 '2000718627', '2000761282', '2001387589', '2001356979', '2000876289']
Length: 155749, dtype: string

In [ ]:
ejemplares = pd.read_parquet(ORIGEN_LOCAL/'ejemplares.parquet',
                             engine="pyarrow",
                             columns=['id_ejemplar', 'id_registro'])
registros = pd.read_parquet(ORIGEN_LOCAL/'registros.parquet',
                            engine="pyarrow", columns=['id_registro', 'titulo', 'autor', 'anio_edicion'])

In [ ]:

COL_BARCODE = "prbarc"
COL_AUTOR = "tiauto_imp"

# 1. Autor del catálogo por código de barras (código -> autor normalizado).
#    validate: avisa si un ejemplar apunta a más de un registro.
autor_cat = (ejemplares[["id_ejemplar", "id_registro"]]
             .merge(registros[["id_registro", "autor"]], on="id_registro",
                    how="left", validate="many_to_one")
             .set_index("id_ejemplar")["autor"])


# 2. ANTES DE RELLENAR: ¿coinciden los formatos?
#    Tomamos préstamos que SÍ tienen autor y comparamos con el del catálogo.
m = (df_prestamos.loc[df_prestamos[COL_AUTOR].notna(), [COL_BARCODE, COL_AUTOR]]
              .drop_duplicates(COL_BARCODE)
              .sample(5000, random_state=42))
m["autor_cat"] = m[COL_BARCODE].map(autor_cat)
m = m.dropna(subset=["autor_cat"])
print(f"Coinciden: {(m[COL_AUTOR] == m['autor_cat']).mean() * 100:.1f} % de {len(m):,}")
print(m[m[COL_AUTOR] != m["autor_cat"]].head(20))   # mira CÓMO difieren

Coinciden: 97.3 % de 4,022
              prbarc                                tiauto_imp  \
4293340      6446604              romanones, aline, condesa de   
3784067   1400365864          integracion de stefano, cristina   
7535551   2000535305                     gary friedrich, autor   
2446830      7454205                     shiel, mattheu phipps   
3092834    101512622                          gonzalez, susana   
289033       7279959                           ofogo, boniface   
8371119      3025692                                     rimas   
1822160      6821748               los infortunios de la virtu   
5817918   1401019647           pardo bazan, emilia, condesa de   
7652146      5485878                dinesen, isakdinesen, isak   
12884897   100312487                    teresa de jesus, santa   
1993562      7759939                     lee shetterly, margot   
2477007      7645277                      el chico burbuja aut   
6966475      5634818  chateaubriand, francois ren

In [ ]:

# Términos que pueden aparecer al final del autor: función (autor, aut., ed., il., trad.)
# y títulos nobiliarios o tratamientos. Amplía la lista con lo que veas en tus datos.
TERMINOS = (r"autor|aut|ed|il|trad|sir|santa"
            r"condes[ao] de|vicomte de|marques[a]? de|duque(?:sa)? de")
PATRON_FINAL = re.compile(rf"[,\s]*\b(?:{TERMINOS})\.?\s*$")

def quitar_sufijos(s: pd.Series) -> pd.Series:
    # Dos pasadas: hay casos con dos términos seguidos ("condesa de, autor")
    for _ in range(2):
        s = s.str.replace(PATRON_FINAL, "", regex=True).str.strip(" ,")
    return s.replace("", pd.NA)

# Quito sufijos tanto de df_prestamos como de catálogo
df_prestamos[COL_AUTOR] = quitar_sufijos(df_prestamos[COL_AUTOR])
autor_cat = quitar_sufijos(autor_cat)

In [ ]:
# 1. Autor del catálogo para cada préstamo.
#   queda vacío si el registro del catálogo no tiene autor.
autor_cat_pres = df_prestamos[COL_BARCODE].map(autor_cat)

usar_cat = autor_cat_pres.notna()          # hay autor en el catálogo -> manda el catálogo
tenia = df_prestamos[COL_AUTOR].notna()       # el préstamo traía autor propio

# 2. Recuentos ANTES de sobrescribir (para el registro de limpieza)
sustituidos = usar_cat & tenia & autor_cat_pres.ne(df_prestamos[COL_AUTOR])
print(f"Préstamos con autor distinto, sustituido por el del catálogo: {sustituidos.sum():,}")
print(f"Préstamos sin autor, rellenados desde el catálogo: {(usar_cat & ~tenia).sum():,}")

# 3. Origen de cada autor: catálogo > préstamos > ninguno
df_prestamos["fuente_autor"] = pd.Categorical(
    np.select([usar_cat, tenia], ["catalogo", "df_prestamos"], default="ninguna"),
    categories=["catalogo", "df_prestamos", "ninguna"])

# 4. Sobrescribir: el autor del catálogo donde lo hay; si no, se queda el del préstamo.
#    Nunca se sustituye un autor del préstamo por un vacío.
df_prestamos[COL_AUTOR] = autor_cat_pres.where(usar_cat, df_prestamos[COL_AUTOR])
del autor_cat_pres                          # libera memoria (18 M de filas)

print(df_prestamos["fuente_autor"].value_counts())

Préstamos con autor distinto, sustituido por el del catálogo: 257,471
Préstamos sin autor, rellenados desde el catálogo: 6,451
fuente_autor
catalogo        14068164
df_prestamos     3206086
ninguna           808946
Name: count, dtype: int64


In [ ]:
sin_autor = df_prestamos["fuente_autor"].eq("ninguna")
cruza = df_prestamos[COL_BARCODE].isin(autor_cat.index)
print(pd.crosstab(sin_autor, cruza, margins=True))

prbarc          False      True       All
fuente_autor                             
False         3176033  14098217  17274250
True           199097    609849    808946
All           3375130  14708066  18083196


In [ ]:
SIN_AUTOR = "[sin autor]"

# Comprobación: el texto fijo no puede coincidir con un autor real
assert not df_prestamos[COL_AUTOR].eq(SIN_AUTOR).any()

# Indicador para filtrar y declarar estas obras (ya está en fuente_autor, pero es más claro)
df_prestamos["sin_autor"] = df_prestamos["fuente_autor"].eq("ninguna")

# Clave: título + autor; si no hay autor, título + "[sin autor]" = en la práctica, solo título
df_prestamos[COL_AUTOR] = df_prestamos[COL_AUTOR].fillna(SIN_AUTOR).astype("category")

print(df_prestamos["sin_autor"].sum(), df_prestamos[COL_AUTOR].eq(SIN_AUTOR).sum())  # deben coincidir: 808.946

808946 808946


### Nulos ISBN

Tanto los ISBN nulos registrados, como los ISBN que no tienen longitud de 13 o 10 dígitos, los cuales se pasan a nulo.

In [ ]:
# Revisión tipo dato y nulos tiisxn longitud

print(df_prestamos['tiisxn_longitud'].dtypes, df_prestamos['tiisxn_longitud'].isna().sum())

float64 3211533


In [ ]:
# Conversión a tipo int y comprobación nulos tiisxn longitud
df_prestamos['tiisxn_longitud'] = df_prestamos['tiisxn_longitud'].astype('Int64')

In [ ]:
# Comprobación cambios:
print(df_prestamos['tiisxn_longitud'].dtypes, df_prestamos['tiisxn_longitud'].isna().sum())


Int64 3211533


In [ ]:
# Pasar a nulo códigos ISBN que no tengan una longitud de 10 o 13 dígitos

len_digitos_ok = [10, 13]

mask_f_nulos = df_prestamos['tiisxn_longitud'].isin(len_digitos_ok)

# Pasa a nulo todos los que no estén en la lista:
df_prestamos['isbn_norm'] = df_prestamos['tiisxn']
df_prestamos.loc[~mask_f_nulos, 'isbn_norm'] = pd.NA

In [ ]:
# Creación nuevo campo con len válidos
df_prestamos['isbn_norm_longitud'] = df_prestamos['isbn_norm'].str.len()

In [ ]:
# Comprobación cambios nulos en isbn_norm_longitud:
print(df_prestamos['isbn_norm_longitud'].dtypes, df_prestamos['isbn_norm_longitud'].isna().sum())

object 3658021


In [ ]:
df_prestamos['isbn_norm_longitud'].unique()

array([<NA>, 10, 13], dtype=object)

In [ ]:
# Conversión a tipo Int y comprobación nulos tiisxn longitud
df_prestamos['isbn_norm_longitud'] = df_prestamos['isbn_norm_longitud'].astype('Int64')

# Comprobación dtype y nulos en isbn_norm_longitud:
print(df_prestamos['isbn_norm_longitud'].dtypes, df_prestamos['isbn_norm_longitud'].isna().sum())

Int64 3658021


**Tratamiento nulos**

In [ ]:

# Máscara booleana de códigos isbn nulos
nulos_isbn = df_prestamos['isbn_norm'].isna()

# Se extraen los prbarc de los registros con isbn nulos

# DF de isbns nulos:
isbn_nulos = df_prestamos.loc[nulos_isbn]

# Prbarc únicos en isbn nulos:
prbar_isbn_nulos = isbn_nulos['prbarc'].unique()
print(f"Codigos unicos en isbn_nulos {len(prbar_isbn_nulos)}")
print(prbar_isbn_nulos)

Codigos unicos en isbn_nulos 1212608
['207904' '225462' '236383' ... '2000043302' '2000797446' '1402336181']


Se han encontrado 1.212.608 códigos prbarc únicos en los 3.658.021 registros con ISBN nulo.

Se buscan coincidencias según código del ejemplar para asignar valor en los registros. Para ello se analiza si valores únicos y coincidentes entre tiisxn y prbarc.

In [ ]:
# Se ve si hay más de un isbn por código de barras del ejemplar.

conflictos = df_prestamos.groupby('prbarc')['isbn_norm'].nunique()
prbarc_con = conflictos[conflictos > 1].index
isbn_prbar = conflictos.value_counts()
print(f"Hay {len(prbarc_con)} códigos de ejemplar con más de un isbn asignado")
print(f"Para cada código de ejemplar se pueden encontrar hasta "
f"{isbn_prbar} códigos de isbn distintos. ")

Hay 97169 códigos de ejemplar con más de un isbn asignado
Para cada código de ejemplar se pueden encontrar hasta isbn_norm
1     2186674
0      363613
2       89110
3        4859
4        1574
5         761
6         449
7         235
8          83
9          47
10         16
12         12
11         12
13          9
14          2
Name: count, dtype: int64 códigos de isbn distintos. 


Se encuentra que hay 97.169 códigos de ejemplar con más de un código ISBN asignado.

Entre los cuales, 363.613 códigos de ejemplares no tienen isbn asignado en ningún registro.



In [ ]:
# Detalle: cada isbn y cuántas veces aparece por prbarc

detalle = (df_prestamos[df_prestamos['prbarc'].isin(prbarc_con)]
           .groupby(['prbarc', 'tiisxn'])
           .size()
           .reset_index(name='n')
           .sort_values(['prbarc', 'n'], ascending=[True, False]))
detalle['n_isbn'] = detalle.groupby('prbarc')['tiisxn'].transform('nunique')
detalle['pct'] = detalle['n'] / detalle.groupby('prbarc')['n'].transform('sum')
detalle.tail(5)

,prbarc,tiisxn,n,n_isbn,pct
211150,9539777,9788413142166,5,2,0.238095
211153,9539932,9788417747312,6,2,0.666667
211152,9539932,8417747311,3,2,0.333333
211155,9539972,9788423355068,11,2,0.687500
211154,9539972,8423355063,5,2,0.312500


A raíz de la muestra, se puede creer que la variación de los códigos isbn entre mismos códigos de ejemplar se debe a la corrección de los mismos, estableciendo la longitud correcta para ellos (13 o 10 dígitos).

Se asigna código ISBN para los distintos registros en los que NO haya más de un código ISBN por código de ejemplar. Ya que he visto, en la revisión manual, que un mismo ejemplar puede tener más de un código ISBN distinto pero válido (13 o 10 dígitos). Pudiendo querer decir que se ha reutilizado el código de ejemplar para una edición distinta del mismo título.

In [ ]:
# Se busca isbn a asignar

# Se establece la base del diccionario de asignación:
# Solo isbn informados:

mask_valido = (df_prestamos['isbn_norm'].notna()
               & df_prestamos['isbn_norm_longitud'].isin(len_digitos_ok))
base_isbn = df_prestamos.loc[mask_valido, ['prbarc', 'isbn_norm']]

# Diccionario asignación:

# Se buscan códigos ejemplar con isbn y
# tomando la primera aparición (en el caso de los que tiene más de una)
# No se tienen en cuenta los prbarc con más de un ISBN.

resumen = base_isbn.groupby('prbarc')['isbn_norm'].agg(['nunique', 'first'])

# Diccionario solo con códigos únicos:
mapa_isbn = resumen.loc[resumen['nunique'] == 1, 'first']
print(f"Ejemplares con ISBN único imputable: {len(mapa_isbn)}")
print(f"Ejemplares descartados (>1 ISBN válido): {(resumen['nunique'] > 1).sum()}")


Ejemplares con ISBN único imputable: 2186674
Ejemplares descartados (>1 ISBN válido): 97169


In [ ]:
# Asignación sobre nulos:

mask_nulos = df_prestamos['isbn_norm'].isna()
imputado = df_prestamos.loc[mask_nulos, 'prbarc'].map(mapa_isbn)

# Registro de ISBN asignado:
df_prestamos['isbn_imputado'] = False
df_prestamos.loc[imputado.dropna().index, 'isbn_imputado'] = True
df_prestamos.loc[mask_nulos, 'isbn_norm'] = imputado

# Se añaden longitudes:
df_prestamos['isbn_norm_longitud'] = df_prestamos['isbn_norm'].str.len()

# Comprobaciones:
print(f"Registros imputados: {df_prestamos['isbn_imputado'].sum()}")
print(f"ISBN nulos restantes: {df_prestamos['isbn_norm'].isna().sum()}")

Registros imputados: 2534194
ISBN nulos restantes: 1123827


Siguen quedando 1.123.827 valores nulos en el campo ISBN. Se revisará con tabla registros de ser necesario.

### Nulos prcoc1

Se intenta asignar valor a todos los registros que compartan código ejemplar con los códigos de los que disponemos información. Extraída de los préstamos realizados a partir de julio del 2025, momento en el cual se empezó a facilitar dicha información.

Nulos iniciales 15.101.765

In [ ]:
nulos_prcoc = df_prestamos['prcoc1'].isna()

# Se extraen los prbarc de los registros con nulos

# DF de prcoc1 nulos:
codigos_nulos = df_prestamos.loc[nulos_prcoc]

# Prbarc únicos en prcoc nulos:
prbar_titulos_nulos = codigos_nulos['prbarc'].unique()
print(f"Codigos prbarc unicos prcoc1  {len(prbar_titulos_nulos)}")

Codigos prbarc unicos prcoc1  2431956


In [ ]:
# prbarc con más de un prcoc1 distinto
# Detalle: cada prcoc1 y cuántas veces aparece por prbarc

detalle = (df_prestamos[df_prestamos['prbarc'].isin(prbarc_con)]
           .groupby(['prbarc', 'prcoc1'], observed=True)
           .size()
           .reset_index(name='n')
           .sort_values(['prbarc', 'n'], ascending=[True, False]))
detalle['n_codigos'] = detalle.groupby('prbarc')['prcoc1'].transform('nunique')
detalle['pct'] = detalle['n'] / detalle.groupby('prbarc')['n'].transform('sum')
detalle.head(5)

,prbarc,prcoc1,n,n_codigos,pct
0,100001348,adu,2,1,1.0
1,100002849,inf,1,1,1.0
2,100002850,inf,1,1,1.0
3,100002853,inf,1,1,1.0
4,100006100,adu,1,1,1.0


In [ ]:
# Creación diccionario

dicc = (df_prestamos.dropna(subset=['prcoc1'])
        .drop_duplicates('prbarc')
        .set_index('prbarc')['prcoc1'])

# Asignación

print(f"Nulos iniciales: {df_prestamos['prcoc1'].isna().sum()}")

df_prestamos['prcoc1_imp'] = df_prestamos['prcoc1']
df_prestamos['prcoc1_imp'] = df_prestamos['prcoc1'].fillna(df_prestamos['prbarc'].map(dicc))

print(f"Nulos finales: {df_prestamos['prcoc1_imp'].isna().sum()}")

Nulos iniciales: 15101740
Nulos finales: 6934156


### Se revisan tipos de dato

In [ ]:
df_prestamos['isbn_norm'].unique()

array(['8484641007', nan, '9788435008952', ..., '9788419879097',
       '9791387512262', '9791387641634'], shape=(320658,), dtype=object)

In [ ]:
df_prestamos['isbn_norm'].isna().sum()

np.int64(1123827)

In [ ]:
print(df_prestamos['isbn_norm'].dtype)                                   # dtype declarado
print(pd.api.types.infer_dtype(df_prestamos['isbn_norm'], skipna=True))  # qué contiene realmente
df_prestamos['isbn_norm'].map(type).value_counts()                # tipos Python mezclados

string
string


,count
isbn_norm,
<class 'str'>,17030001
<class 'pandas._libs.missing.NAType'>,1123827


In [ ]:
df_prestamos['isbn_norm_longitud'] = df_prestamos['isbn_norm_longitud'].astype('Int64')

In [ ]:
df_prestamos.isna().sum()

,0
prbarc,0
prprsu,0
prcolp,0
prcocp,0
prlebi,0
prlesu,129561
pradul,0
prfpre,0
prcocs,0
tititu,65


### Eliminan registros con código prcolp distintos a LEC, LECI, ALU, PRO, ROB, INV, INS o EVE.

Solo se mantienen registros con estos códigos de lector ya que son los que identifican a los usuarios con préstamos individuales.

In [ ]:
df_prestamos['prcolp'].value_counts()

,count
prcolp,
lec,14650805
leci,3361617
tel,40015
alu,29061
ent,27737
pro,19815
rob,18454
inv,3170
dig,980


In [ ]:
print(f"Registros iniciales: {len(df_prestamos)}")

codigos_no_indv = ['tel', 'ent', 'dig', 'pco', 'pin', 'bib', 'c19','xcua']
registros_no_indv = df_prestamos['prcolp'].isin(codigos_no_indv)
print(f"Registros a eliminar: {registros_no_indv.sum()}")

df_prestamos = df_prestamos.loc[~registros_no_indv]

print(f"Registros finales: {len(df_prestamos)}")

Registros iniciales: 18153828
Registros a eliminar: 70632
Registros finales: 18083196


In [ ]:
df_prestamos.shape

(18083196, 22)

### Eliminación de columnas

Columnas que no se exportan en versión final

- prcocp -> tipo de ejemplar prestado. Código único, PRL.
- prcocs -> tipo soporte prestado. Código único, LIB.
- tititu -> Títulos previos a la asignación normalizada.
- tiauto -> Autores previos a la asignación normalizada.
- tiisxn -> Códigos ISBN previos a la anulación por len no válido y asignación posterior.
- prcoc1 -> Códigos tipo lector previos a la asignación realizada.
- tiisxn_longitud -> Longitudes código ISBN previas.
- origen_titulo -> Registro tipo asignación
- origen_autor -> Registro tipo asignación
- isbn_norm_longitud -> Longitudes finales
- isbn_imputado -> Si código imputado o no
- prlesu -> Código biblioteca donde se realizó el carnet. Muchos nulos, códigos sin trazabilidad por ser fuera de la red de bibliotecas públicas. Se deja el campo red_carnet (prlebi) del cual hay datos para todos los registros.
- fuente_autor
- sin_autor

In [ ]:
df_prestamos.columns

Index(['prbarc', 'prprsu', 'prcolp', 'prcocp', 'prlebi', 'prlesu', 'pradul',
       'prfpre', 'prcocs', 'tititu', 'tiauto', 'tiisxn', 'prcoc1',
       'tiisxn_longitud', 'tititu_imp', 'origen_titulo', 'tiauto_imp',
       'origen_autor', 'isbn_norm', 'isbn_norm_longitud', 'isbn_imputado',
       'prcoc1_imp', 'fuente_autor', 'sin_autor'],
      dtype='object')

In [ ]:
prestamos = df_prestamos[['prbarc', 'prprsu', 'prcolp', 'prlebi', 'pradul',
       'prfpre', 'tititu_imp', 'tiauto_imp','isbn_norm', 'prcoc1_imp']]

In [ ]:
# Nuevos nombre columnas
prestamos = prestamos.rename(columns= {
    'prbarc': 'id_ejemplar',
    'prprsu': 'id_biblioteca',
    'prcolp': 'id_lector',
    'prlebi': 'id_red',
    'pradul': 'carnet_adulto',
    'prfpre': 'fecha_prestamo',
    'tititu_imp': 'titulo',
    'tiauto_imp': 'autor',
    'isbn_imputado': 'isbn',
    'prcoc1_imp': 'id_publico'
    })

In [ ]:
prestamos.dtypes

,0
id_ejemplar,string[pyarrow]
id_biblioteca,category
id_lector,category
id_red,category
carnet_adulto,category
fecha_prestamo,datetime64[ns]
titulo,string[pyarrow]
autor,string[pyarrow]
isbn_norm,string[pyarrow]
id_publico,category


# Cruce Préstamos y Registros

In [ ]:
# Cruce de prestamos con ejemplares
df_prestamos["cruza"] = df_prestamos["id_ejemplar"].isin(ejemplares["id_ejemplar"])

# % De id_ejemplar en  catálogo por año
df_prestamos["anio"] = pd.to_datetime(df_prestamos["fecha_prestamo"]).dt.year
tasa_a = df_prestamos.groupby("anio")["cruza"].mean() * 100
print(tasa_a)


# % de cruce sobre todos los ejemplares distintos.
unicos = df_prestamos.drop_duplicates("id_ejemplar")
print(unicos["cruza"].mean() * 100)

anio
2018    65.360809
2019    67.650999
2020    71.317416
2021    76.162107
2022    79.822386
2023    84.329437
2024    89.035385
2025    95.082688
2026    95.231914
Name: cruza, dtype: float64
80.69879213449124


# Asignar primer año edición

In [ ]:
# Cálculo de obras con año de edición:

registros['anio_edicion'] = registros['anio_edicion'].astype('string')
anio_reg = registros.set_index('id_registro')['anio_edicion']
# Si anio_edicion está guardado como Period, pasarlo a entero sin perder los nulos
if isinstance(anio_reg.dtype, pd.PeriodDtype):
    anio_reg = anio_reg.dt.to_timestamp().dt.year
anio_reg = anio_reg.astype('Int16')

# Un código de barras = un ejemplar (si no, map fallaría)
assert not ejemplares['id_ejemplar'].duplicated().any()
anio_por_ejemplar = ejemplares.set_index('id_ejemplar')['id_registro'].map(anio_reg)

# --- 2. Año de edición del ejemplar de cada préstamo (columna temporal) ------
anio_ej = df_prestamos['id_ejemplar'].map(anio_por_ejemplar).astype('Int16')

# --- 3. Clave de obra como número entero (título + autor) --------------------
cod_t = df_prestamos['titulo'].astype('category').cat.codes.astype('int64')
cod_a = df_prestamos['autor'].astype('category').cat.codes.astype('int64')
clave = (cod_t * (cod_a.max() + 1) + cod_a).to_numpy()

df_prestamos['anio_obra'] = anio_ej.groupby(clave).transform('min')
df_prestamos['obra_cubierta'] = df_prestamos['anio_obra'].notna()

print('Tasa C por año (%):')
print((df_prestamos.groupby('anio_prestamo')['obra_cubierta'].mean() * 100).round(2))


Tasa C por año (%):
anio_prestamo
2018    97.40
2019    97.76
2020    98.26
2021    98.63
2022    98.91
2023    99.16
2024    99.41
2025    99.70
2026    98.13
Name: obra_cubierta, dtype: float64


In [ ]:
prestamos.columns

Index(['id_ejemplar', 'id_biblioteca', 'id_lector', 'id_red', 'carnet_adulto',
       'fecha_prestamo', 'titulo', 'autor', 'isbn_norm', 'id_publico',
       'anio_obra', 'obra_cubierta', 'anio_prestamo', 'fuente_autor',
       'sin_autor'],
      dtype='object')

In [ ]:
prestamos = prestamos.drop(columns=['obra_cubierta', 'fuente_autor','sin_autor'])

# Exportación parquet

In [ ]:
df_prestamos.to_parquet(
    DRIVE_DESTINO / 'prestamos.parquet',
    engine="pyarrow",
    compression="snappy",
    index=False,
)